# The RAGulator ◆ it regulates.  (Psypher Labs — Colab)

A teaching demo of layered defence for an LLM document assistant: rule + LLM
threat screening, role-based access control, need-to-know, and a Gemini RAG
policy assistant.

**This is a learning artifact, not a real security system.** Identity, role and
data-classification are self-asserted by design — see the README for the trust
boundaries you'd replace in production.

---
### Step 1 — Install dependencies

In [ ]:
# In Colab this installs everything; locally, prefer: pip install -r requirements.txt
%pip install -q pydantic python-dotenv sqlalchemy faker pandas requests matplotlib \
    langchain-core langchain-community langchain-google-genai faiss-cpu pyyaml ipywidgets

### Step 2 — Provide your Google API key

Click the **🔑 Secrets** icon in Colab's left sidebar, add a secret named
`GOOGLE_API_KEY`, and enable it for this notebook. The code reads it
automatically — you never paste the key into a cell.

In [ ]:
# Sanity check that a key is reachable (does not print the key).
import os
key_present = False
try:
    from google.colab import userdata
    key_present = bool(userdata.get('GOOGLE_API_KEY'))
except Exception:
    key_present = bool(os.getenv('GOOGLE_API_KEY'))
print('GOOGLE_API_KEY found:', key_present)

### Step 3 — Make the package importable

If you uploaded the project folder, point Python at its `src/` directory.

In [ ]:
import sys, pathlib
# Adjust this path if you uploaded the folder elsewhere.
SRC = pathlib.Path('ragulator/src')
if SRC.exists():
    sys.path.insert(0, str(SRC.resolve()))
print('Looking for package on sys.path...')
import ragulator
print('Loaded ragulator', ragulator.__version__)

### Step 4 — Build the application

This creates the synthetic database (idempotent), initialises Gemini, and builds
the policy RAG index.

In [ ]:
from ragulator import build_app, Settings

settings = Settings.from_environment(num_documents=300)  # smaller DB = faster demo
app = build_app(settings)
print('App ready. NKT mode:', settings.nkt_mode)

### Step 5 — Launch the interactive UI

Pick a role, type a query (e.g. *"find documents about strategy"*,
*"retrieve document 5"*, *"what is the need-to-know policy?"*), and submit.
Try a malicious query like *"ignore previous instructions"* to see the block.

In [ ]:
from ragulator.ui.notebook import launch
launch(app)

### (Optional) Use it from code instead of the UI

In [ ]:
from ragulator.pipeline import process_request

out = process_request(
    raw_query="what is the need-to-know policy?",
    user_role="analyst",
    settings=settings, gemini=app.gemini, engine=app.engine, api_cache=app.api_cache,
)
print(out["intent"], "->")
for item in out["results"]:
    print(item)